# Preconditioned GAMERA/ReMIX workflow

This notebook captures the workflow we want for repeatable GAMERA case studies:

1. Build a clean run directory.
2. Run a constant solar-wind preconditioning interval long enough for the magnetosphere/ionosphere to settle.
3. Verify the ReMIX output and checkpoint files.
4. Reuse the late preconditioned restart as the initial condition for event/case-study solar wind.
5. Plot FAC and horizontal Hall-current diagnostics without doing any SECS inversion.

The completed 3-hour shallow GAMERA+ReMIX test is the reference run for this notebook. It reached `Step#181` at `10800.04 s` and produced a stable southward-IMF convection state.

## Assumptions

- Work from `/home/simon/gits/SECpy_cleaned`.
- Use the `SECSpy` conda environment for runtime and plotting.
- Use the locally built Kaiju executable under `gamera_ireland/external/kaiju-master`.
- The current validated fast path is **shallow GAMERA+ReMIX**: `doDeep="F"`, no RAIJU block.
- For shallow coupling, Kaiju requires `REMIX/precipitation aurora_model_type="FEDDER"` or `"SUNNY"`; the setup script handles this when `--no-raiju` is used.
- Use a fresh run directory for every real run. Reusing an old run directory can produce HDF5 `name already exists` diagnostics.

In [ ]:
from pathlib import Path
import json
import re
import xml.etree.ElementTree as ET

PROJECT = Path('/home/simon/gits/SECpy_cleaned')
GI = PROJECT / 'gamera_ireland'
SIMS = GI / 'simulations'
FIGS = GI / 'output' / 'figures'
KAIJU = GI / 'external' / 'kaiju-master'

PRECON_RUNID = 'precon_southward_bz10_3h_shallow'
PRECON_DIR = SIMS / PRECON_RUNID

REFERENCE_RUNID = 'lompe_southward_3h_remix_shallow'
REFERENCE_DIR = SIMS / REFERENCE_RUNID

print(PROJECT)
print(KAIJU / 'build_serial/bin/voltron.x')

## 1. Check the existing completed preconditioning run

This uses the monitor script instead of assuming any dataset names beyond the Kaiju `Step#` group convention.

In [ ]:
!cd {PROJECT} && conda run -n SECSpy python gamera_ireland/scripts/monitor_mage_run.py \
  gamera_ireland/simulations/{REFERENCE_RUNID} \
  --runid {REFERENCE_RUNID}

In [ ]:
def latest_numeric_restart(run_dir: Path, runid: str, component: str = 'mix'):
    pattern = f'{runid}.{component}.Res.*.h5'
    candidates = []
    for path in run_dir.glob(pattern):
        match = re.search(r'\.Res\.(\d+)\.h5$', path.name)
        if match:
            candidates.append((int(match.group(1)), path))
    if not candidates:
        raise FileNotFoundError(f'No numeric restart files found for {pattern}')
    return max(candidates)

restart_number, restart_path = latest_numeric_restart(REFERENCE_DIR, REFERENCE_RUNID, 'mix')
print(restart_number)
print(restart_path)

## 2. Create a clean preconditioning run directory

This command creates a 3-hour constant southward-IMF preconditioning run. It does **not** run Voltron; it only writes the grid, `bcwind.h5`, XML, calcdb XML and manifest.

The validated parameters are deliberately simple:

- `Vx = -450 km/s`
- `n = 5 cm^-3`
- `Bz = -10 nT`
- `By = 0 nT`
- 1-minute ReMIX output
- 10-minute restart cadence
- shallow GAMERA+ReMIX mode via `--no-raiju`

In [ ]:
setup_precon = f"""
cd {PROJECT} && conda run -n SECSpy python gamera_ireland/scripts/setup_lompe_candidate_run.py \\
  --run-dir gamera_ireland/simulations/{PRECON_RUNID} \\
  --runid {PRECON_RUNID} \\
  --clobber \\
  --duration-s 10800 \\
  --spinup-s 60 \\
  --dtout-s 60 \\
  --cadence-s 60 \\
  --density-base 5 \\
  --density-delta 0 \\
  --pulse-start-s 0 \\
  --pulse-end-s 10800 \\
  --vx-kms -450 \\
  --by-nt 0 \\
  --bz-south-nt -10 \\
  --f107 120 \\
  --kp 4 \\
  --do-starlight \\
  --restart-dt-s 600 \\
  --no-raiju
""".strip()
print(setup_precon)

In [ ]:
# Run this cell only when you want to create/recreate the preconditioning deck.
# !{setup_precon}

## 3. Launch the preconditioning run detached

This lets Voltron continue after the notebook cell returns. The log file can be quiet for long periods because Fortran/stdout buffering is normal; use the progress watcher below for the actual status.

In [ ]:
launch_precon = f"""
cd {PRECON_DIR} && \\
setsid env KAIJUHOME={KAIJU} \\
  conda run --no-capture-output -n SECSpy \\
  {KAIJU / 'build_serial/bin/voltron.x'} {PRECON_RUNID}.xml \\
  > voltron_serial.log 2>&1 < /dev/null &
""".strip()
print(launch_precon)

In [ ]:
# Run this cell only when the deck exists and you want to start the long preconditioning run.
# !bash -lc "{launch_precon}"

## 4. Watch progress

In a normal terminal, use this form so `conda run` does not buffer the progress bar:

In [ ]:
watch_cmd = f"""
cd {PROJECT} && conda run --no-capture-output -n SECSpy python -u \\
  gamera_ireland/scripts/watch_mage_progress.py \\
  gamera_ireland/simulations/{PRECON_RUNID} \\
  --runid {PRECON_RUNID} \\
  --interval-s 30
""".strip()
print(watch_cmd)

The run is ready to use as a preconditioned state when:

- latest ReMIX time is at or beyond `10800 s`,
- the latest restart number is `00018` for a 10-minute cadence over 3 hours,
- the FAC/current pattern has stopped changing dramatically except for normal model variability,
- no Voltron process remains for that run ID.

In [ ]:
# One-shot status check after the run has finished.
# !cd {PROJECT} && conda run -n SECSpy python gamera_ireland/scripts/monitor_mage_run.py gamera_ireland/simulations/{PRECON_RUNID} --runid {PRECON_RUNID}

## 5. Render preconditioning diagnostics

This creates one frame per ReMIX minute, starting at `Step#1` so the `Step#0` spinup state is omitted.

In [ ]:
movie_cmd = f"""
cd {PROJECT} && conda run -n SECSpy python gamera_ireland/scripts/make_remix_fac_hall_movie.py \\
  --mix gamera_ireland/simulations/{PRECON_RUNID}/{PRECON_RUNID}.mix.h5 \\
  --xml gamera_ireland/simulations/{PRECON_RUNID}/{PRECON_RUNID}.xml \\
  --frames-dir gamera_ireland/output/figures/{PRECON_RUNID}_movie \\
  --filename {PRECON_RUNID}_fac_hall \\
  --fps 6 \\
  --start-step 1
""".strip()
print(movie_cmd)

In [ ]:
# Run after the preconditioning run is complete.
# !{movie_cmd}

## 6. Prepare a case-study deck from a preconditioned restart

This is the next stage to validate. Local Kaiju examples use:

```xml
<restart doRes="T" resID="msphereRes" nRes="10"/>
```

under `<Gamera>` to restart from a previous checkpoint. For our preconditioning run, the corresponding restart ID is the preconditioning run ID and the restart number is usually `18` for a 3-hour, 10-minute-cadence run.

The case-study run should use a **new run directory and run ID**, but point its `Gamera/restart` to the preconditioning run ID and restart number. This cell edits only the XML in the new case directory; it assumes you already created the case deck with `setup_lompe_candidate_run.py` or an OMNI-based setup script.

In [ ]:
def set_gamera_restart(xml_path: Path, restart_runid: str, restart_number: int) -> None:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    gamera = root.find('Gamera') or root.find('gamera')
    if gamera is None:
        raise ValueError(f'No Gamera block in {xml_path}')
    restart = gamera.find('restart')
    if restart is None:
        restart = ET.SubElement(gamera, 'restart')
    restart.set('doRes', 'T')
    restart.set('resID', restart_runid)
    restart.set('nRes', str(restart_number))
    tree.write(xml_path, encoding='unicode')

restart_number, restart_path = latest_numeric_restart(REFERENCE_DIR, REFERENCE_RUNID, 'mix')
print(f'Restart candidate: {restart_number} from {restart_path.name}')

In [ ]:
# Example only: after creating a case-study deck, patch it to start from the preconditioned restart.
# CASE_RUNID = 'case_bz_turning_from_precon'
# CASE_DIR = SIMS / CASE_RUNID
# set_gamera_restart(CASE_DIR / f'{CASE_RUNID}.xml', REFERENCE_RUNID, 18)
# print((CASE_DIR / f'{CASE_RUNID}.xml').read_text()[:1200])

## 7. Next validation step

Before using this for May 2024 production studies, run a short restart smoke test:

1. Create a new 10-20 minute case deck with a changed driver, e.g. stronger southward IMF or a By perturbation.
2. Patch `Gamera/restart` to `doRes="T"`, `resID=<preconditioning_runid>`, `nRes=18`.
3. Copy or symlink the preconditioning restart files into the case directory if Kaiju does not find them by `resID` directly.
4. Run Voltron and verify that `Step#1` starts from a developed convection state rather than a cold start.

Once that smoke test works, we can wrap the case-deck creation and restart patching into a dedicated script.